In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/rondii/dataset-selected-features/dataset_selected_features (3).pkl


In [10]:
df = pd.read_pickle('/kaggle/input/datasets/rondii/dataset-selected-features/dataset_selected_features (3).pkl')

In [11]:
import optuna
import lightgbm as lgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

In [12]:
from sklearn.model_selection import train_test_split

n = len(df)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_df = df.iloc[:train_end]
val_df = df.iloc[train_end:val_end]
test_df = df.iloc[val_end:]

X_train_full = train_df.drop(columns=['traffic_volume'])
y_train_full = train_df['traffic_volume']

X_val = val_df.drop(columns=['traffic_volume'])
y_val = val_df['traffic_volume']

X_test = test_df.drop(columns=['traffic_volume'])
y_test = test_df['traffic_volume']

In [5]:
tscv = TimeSeriesSplit(n_splits=5)

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000, step=100),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'subsample': trial.suggest_float('subsample', 0.8, 1.0),
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'n_jobs': -1,
        'verbose': -1
    }

    scores = []
    for train_idx, val_idx in tscv.split(X_train_full):
        X_tr, X_val_cv = X_train_full.iloc[train_idx], X_train_full.iloc[val_idx]
        y_tr, y_val_cv = y_train_full.iloc[train_idx], y_train_full.iloc[val_idx]

        model = lgb.LGBMRegressor(**params)
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val_cv, y_val_cv)],
            callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False)]
        )
        pred = model.predict(X_val_cv)
        scores.append(mean_absolute_error(y_val_cv, pred))

    return np.mean(scores)

study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f"Лучший MAE: {study.best_value:.2f}")
print(f"Лучшие параметры: {study.best_params}")

[I 2026-08-15 13:10:36,566] A new study created in memory with name: no-name-f2047954-3ecf-44fe-b0d8-65c55a77e655


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-08-15 13:10:38,335] Trial 0 finished with value: 276.63644252403753 and parameters: {'n_estimators': 400, 'max_depth': 10, 'learning_rate': 0.08960785365368121, 'subsample': 0.9197316968394074, 'num_leaves': 40}. Best is trial 0 with value: 276.63644252403753.
[I 2026-08-15 13:10:39,257] Trial 1 finished with value: 292.282455841785 and parameters: {'n_estimators': 200, 'max_depth': 3, 'learning_rate': 0.13394334706750485, 'subsample': 0.9202230023486417, 'num_leaves': 112}. Best is trial 0 with value: 276.63644252403753.
[I 2026-08-15 13:10:40,383] Trial 2 finished with value: 277.7905982995113 and parameters: {'n_estimators': 100, 'max_depth': 10, 'learning_rate': 0.12106896936002161, 'subsample': 0.8424678221356553, 'num_leaves': 43}. Best is trial 0 with value: 276.63644252403753.
[I 2026-08-15 13:10:41,979] Trial 3 finished with value: 281.6652495929843 and parameters: {'n_estimators': 200, 'max_depth': 5, 'learning_rate': 0.048164145309070844, 'subsample': 0.8863890037284

In [6]:
X_train_val = pd.concat([X_train_full, X_val])
y_train_val = pd.concat([y_train_full, y_val])

best_model = lgb.LGBMRegressor(**study.best_params, n_jobs=-1, verbose=-1)
best_model.fit(X_train_val, y_train_val)

LGBMRegressor(learning_rate=0.041453526245942264, max_depth=9, n_estimators=200,
              n_jobs=-1, num_leaves=68, subsample=0.9304784553226552,
              verbose=-1)

In [7]:
y_pred = best_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100
smape = np.mean(np.abs(y_test - y_pred) / ((np.abs(y_test) + np.abs(y_pred)) / 2)) * 100

print(f"MAE:   {mae:.2f}")
print(f"RMSE:  {rmse:.2f}")
print(f"MAPE:  {mape:.2f}%")
print(f"SMAPE: {smape:.2f}%")
print(f"R²:    {r2:.4f}")

MAE:   210.66
RMSE:  336.96
MAPE:  10.28%
SMAPE: 9.39%
R²:    0.9711


In [5]:
import torch
from torch.utils.data import Dataset, DataLoader

In [13]:
class TrafficDataset(Dataset):
    def __init__(self, X, y, window_size):
        self.X = X.values.astype('float32') if hasattr(X, 'values') else X.astype('float32')
        self.y = y.values.astype('float32') if hasattr(y, 'values') else y.astype('float32')
        self.window_size = window_size

    def __len__(self):
        return len(self.X) - self.window_size

    def __getitem__(self, idx):
        return torch.tensor(self.X[idx : idx + self.window_size]), torch.tensor(self.y[idx + self.window_size])

In [7]:
import torch.nn as nn

In [14]:
class LSTMModel(nn.Module):
    def __init__(self, n_features, hidden_size, n_layers, dropout):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=n_features,
            hidden_size=hidden_size,
            num_layers=n_layers,
            dropout=dropout if n_layers > 1 else 0.0,
            batch_first=True
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        lstm_out, (h_n, c_n) = self.lstm(x)
        last_output = lstm_out[:, -1, :]
        out = self.dropout(last_output)
        out = self.fc(out)
        return out.squeeze(-1)

In [15]:
n_features = X_train_full.shape[1]

In [13]:
def objective_lstm(trial):
    hidden_size = trial.suggest_int('hidden_size', 50, 128, step=16)
    n_layers = trial.suggest_int('n_layers', 1, 2)
    dropout = trial.suggest_float('dropout', 0.2, 0.3)
    lr = trial.suggest_float('lr', 0.001, 0.1, log=True)
    batch_size = trial.suggest_categorical('batch_size', [16, 32, 64])
    window_size = trial.suggest_categorical('window_size', [10, 20, 30])
    train_ds = TrafficDataset(X_train_full, y_train_full, window_size)
    val_ds = TrafficDataset(X_val, y_val, window_size)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=False)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    model = LSTMModel(n_features=n_features, hidden_size=hidden_size, n_layers=n_layers, dropout=dropout)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.L1Loss()

    best_val_loss = float('inf')
    epochs_no_improve = 0
    max_epochs = 150

    for epoch in range(max_epochs):
        model.train()
        for x_batch, y_batch in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(x_batch), y_batch)
            loss.backward()
            optimizer.step()

        model.eval()
        val_losses = []
        with torch.no_grad():
            for x_batch, y_batch in val_loader:
                val_losses.append(criterion(model(x_batch), y_batch).item())
        val_loss = np.mean(val_losses)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= 3:
                break

    return best_val_loss

study_lstm = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
study_lstm.optimize(objective_lstm, n_trials=15, show_progress_bar=True)

print(f"Лучший val MAE: {study_lstm.best_value:.2f}")
print(f"Лучшие параметры: {study_lstm.best_params}")

[I 2026-08-15 13:13:52,525] A new study created in memory with name: no-name-f792bcb2-0d5a-43ce-9a57-b1f5aa34f5fb


  0%|          | 0/15 [00:00<?, ?it/s]

/tmp/ipykernel_58/1710093241.py:2: UserWarning: The distribution is specified by [50, 128] and step=16, but the range is not divisible by `step`. It will be replaced with [50, 114].
  hidden_size = trial.suggest_int('hidden_size', 50, 128, step=16)


[I 2026-08-15 13:18:53,682] Trial 0 finished with value: 1717.3611001773756 and parameters: {'hidden_size': 66, 'n_layers': 2, 'dropout': 0.2731993941811405, 'lr': 0.015751320499779727, 'batch_size': 16, 'window_size': 10}. Best is trial 0 with value: 1717.3611001773756.


/tmp/ipykernel_58/1710093241.py:2: UserWarning: The distribution is specified by [50, 128] and step=16, but the range is not divisible by `step`. It will be replaced with [50, 114].
  hidden_size = trial.suggest_int('hidden_size', 50, 128, step=16)


[I 2026-08-15 13:24:31,116] Trial 1 finished with value: 700.2321733354448 and parameters: {'hidden_size': 50, 'n_layers': 2, 'dropout': 0.28324426408004216, 'lr': 0.0026587543983272706, 'batch_size': 64, 'window_size': 10}. Best is trial 1 with value: 700.2321733354448.
[I 2026-08-15 13:30:16,330] Trial 2 finished with value: 936.0755015980113 and parameters: {'hidden_size': 98, 'n_layers': 1, 'dropout': 0.22921446485352182, 'lr': 0.005404103854647328, 'batch_size': 32, 'window_size': 20}. Best is trial 1 with value: 700.2321733354448.
[I 2026-08-15 13:31:35,096] Trial 3 finished with value: 1716.3488758433948 and parameters: {'hidden_size': 98, 'n_layers': 1, 'dropout': 0.20650515929852797, 'lr': 0.07902619549708234, 'batch_size': 16, 'window_size': 20}. Best is trial 1 with value: 700.2321733354448.
[I 2026-08-15 13:32:22,270] Trial 4 finished with value: 1718.345972511985 and parameters: {'hidden_size': 50, 'n_layers': 1, 'dropout': 0.20343885211152185, 'lr': 0.06586289317583113, '

In [17]:
def train_lstm_simple(model, train_loader, lr, max_epochs=150, patience=3, min_delta=1.0):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.L1Loss()

    best_loss = float('inf')
    epochs_no_improve = 0

    for epoch in range(max_epochs):
        model.train()
        epoch_losses = []
        for x_batch, y_batch in train_loader:
            optimizer.zero_grad()
            preds = model(x_batch)
            loss = criterion(preds, y_batch)
            loss.backward()
            optimizer.step()
            epoch_losses.append(loss.item())

        train_loss = np.mean(epoch_losses)
        print(f"Epoch {epoch+1}, train MAE: {train_loss:.2f}")

        if best_loss - train_loss > min_delta:
            best_loss = train_loss
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Остановка на эпохе {epoch+1}")
                break

    return model

In [18]:
X_train_val = pd.concat([X_train_full, X_val])
y_train_val = pd.concat([y_train_full, y_val])

window_size = 20
batch_size = 64

train_val_dataset = TrafficDataset(X_train_val, y_train_val, window_size)
train_val_loader = DataLoader(train_val_dataset, batch_size=batch_size, shuffle=False)

n_features = X_train_full.shape[1] 

best_lstm = LSTMModel(
    n_features=n_features,
    hidden_size=114,
    n_layers=2,
    dropout=0.23308980248526492
)

best_lstm = train_lstm_simple(
    best_lstm,
    train_val_loader,
    lr=0.0013400367243354798,
    max_epochs=100,
    patience=3,
    min_delta=0.5
)

test_dataset = TrafficDataset(X_test, y_test, window_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

best_lstm.eval()
all_preds, all_true = [], []
with torch.no_grad():
    for x_batch, y_batch in test_loader:
        preds = best_lstm(x_batch)
        all_preds.append(preds.numpy())
        all_true.append(y_batch.numpy())

y_pred_lstm = np.concatenate(all_preds)
y_true_lstm = np.concatenate(all_true)

mae_lstm = mean_absolute_error(y_true_lstm, y_pred_lstm)
rmse_lstm = np.sqrt(mean_squared_error(y_true_lstm, y_pred_lstm))
r2_lstm = r2_score(y_true_lstm, y_pred_lstm)
mape_lstm = np.mean(np.abs((y_true_lstm - y_pred_lstm) / y_true_lstm)) * 100
smape_lstm = np.mean(np.abs(y_true_lstm - y_pred_lstm) / ((np.abs(y_true_lstm) + np.abs(y_pred_lstm)) / 2)) * 100

print(f"LSTM — MAE: {mae_lstm:.2f}")
print(f"LSTM — RMSE: {rmse_lstm:.2f}")
print(f"LSTM — MAPE: {mape_lstm:.2f}%")
print(f"LSTM — SMAPE: {smape_lstm:.2f}%")
print(f"LSTM — R²: {r2_lstm:.4f}")

Epoch 1, train MAE: 3214.24
Epoch 2, train MAE: 3117.09
Epoch 3, train MAE: 3021.95
Epoch 4, train MAE: 2934.80
Epoch 5, train MAE: 2861.46
Epoch 6, train MAE: 2795.02
Epoch 7, train MAE: 2733.86
Epoch 8, train MAE: 2674.06
Epoch 9, train MAE: 2616.32
Epoch 10, train MAE: 2562.55
Epoch 11, train MAE: 2513.05
Epoch 12, train MAE: 2467.04
Epoch 13, train MAE: 2421.12
Epoch 14, train MAE: 2376.64
Epoch 15, train MAE: 2334.40
Epoch 16, train MAE: 2279.34
Epoch 17, train MAE: 2166.66
Epoch 18, train MAE: 2090.92
Epoch 19, train MAE: 2014.04
Epoch 20, train MAE: 1953.89
Epoch 21, train MAE: 1895.22
Epoch 22, train MAE: 1842.12
Epoch 23, train MAE: 1783.60
Epoch 24, train MAE: 1733.93
Epoch 25, train MAE: 1683.59
Epoch 26, train MAE: 1632.92
Epoch 27, train MAE: 1576.33
Epoch 28, train MAE: 1529.53
Epoch 29, train MAE: 1483.95
Epoch 30, train MAE: 1433.88
Epoch 31, train MAE: 1393.00
Epoch 32, train MAE: 1347.19
Epoch 33, train MAE: 1315.40
Epoch 34, train MAE: 1272.06
Epoch 35, train MAE: 12